In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib.patches as mpatches


from PIL import Image

# Will be used for the dark spots in the dataset
from scipy import ndimage

# Fits a straight line compared to the longitude vs time
from scipy.stats import lineregress


In [ ]:
IMAGE = "../Data"

# The Dataset size
Days = 9

This gets the center of the Solar Disk and the radius

In [ ]:
'''
Function: Gets the Center of the Sun and calculated the radius of the Sun in the image

Input:
    Image

Return: 
    X coordinate
    Y coordinate
    Radius

Sources:
    CMPUT 206 Notes

'''

def getCenter(img_arr):

    # Since HMI orange images are the strongest in the RED channel
    # Of the image, it will be best used to differentiate
    # The black space and orange Sun

    red_img = img_arr[:,:,0].astype(float)

    threshold = 30

    # Gets a mask to make sure only the sun is considered for center
    # Creates a binary grid:
    # 1: If it is the sun
    # 0: If it is outer space
    mask = red > threshold

    rows = []
    # Check every row which contains a pixel of the sun
    # If a row has a sun in it then we keep the position of the row in an array
    for r in range(mask.shape[0]):
        if 1 in mask[r, :]:
            rows.append(r)
    rows = np.array(rows)

    # Goes through each column checking every pixel
    # Checks if it contains any part of the suns pixel
    # If it does store the positioning
    cols = []
    for c in range(mask.shape[1]):
        if 1 in mask[:,c]:
            cols.append(c)
    cols = np.array(cols)

    row_min = rows[0]
    row_max = rows[-1]

    col_min = cols[0]
    col_max = cols[-1]

    # (Max + Min) / 2 
    # Gets the center coordinates

    y = (row_max + row_min)/ 2.0
    x = (col_max + col_min)/ 2.0

    # Radius
    vertD = row_max - row_min
    horzD = col_max - col_min

    R = (vertD + horzD) / 4.0

    return x,y,R

Gets all the Sunspots and returns the coordinates of the sunspots

In [ ]:
def getSunspots(img_arr, x,y,R):

    # Gets the different color channels RGB for the image
    rImg = img_array[:,:,0].astype(float)
    gImg = img_array[:,:,1].astype(float)
    bImg = img_array[:,:,2].astype(float)

    rows = img_arr.shape[0]
    cols = img_arr.shape[1]

    # Creates a grid same size as image
    # If there are pixels inside the disk its true
    mask = [[False]*cols for i in range(rows)]

    # Checks for each pixel inside the sun
    for r in range(rows):
        for c in range(cols):
            # Use pythagorean theorem to get distance
            # Of the pixel to the center of sun
            vertD = r - y
            horzD = c - x
            dist = np.sqrt(horzD ** 2 + vertD ** 2)

            # Checks if the pixel is within the sun
            # Less than 98% so we dont mistake edges
            if dist < R * 0.98:
                mask[r][c] = True
    
    mask = np.array(mask)

    # Average brightness of each pixel
    avgB = (rImg + gImg + bImg) / 3.0

    # Gets all the brightness values inside the solar image
    insideDisk = avgB[mask]

    # Gets the average of the pixels inside the disk
    meanB = np.mean(insideDisk)

    # Gets standard deviation  on how spread out the brightness is
    stdB = np.std(insideDisk)

    # Gets a threshold to get sunspots
    # Use 2.5 to single out sunspots
    threshold = meanB - 2.5 * stdB

    # Mark whether each pixel is a sunspot and/or not a text
    # Since most of the text is green we calculate a green ratio
    # Add 0.0001 to make sure we dont divide bt 0
    isGreen = gImg / (rImg + gImg + bImg + 0.0001)

    # Checks conditions whether it is a sunspot
    isBlack = avgB < threshold
    # Makes sure that the pixel is inside the disk
    insideBlack = mask
    # Makes sure its not a green label
    notLabel = isGree < 0.5

    # Gets the sunspot mask to get their locations
    sunspotM = isBlack & insideBlack * notLabel

    # Groups all True pixels together and labels them into groups wih a unique lavel
    label, n = ndimage.label(sunspotM)

    sunspot_blob = []

    for i in range(1,n+1):
        # Row and column of eahc pixel inside the sunspot_blob
        brow, bcol = np.where(label == i)
        # Gets number of pixels in the sunspot
        numPixels = len(brow)

        # If the sunspot is too little/ or large we skip it
        # Prevents inaccuracies
        if numPixels < 15 or numPixels > 5000:
            continue
        
        # Gets the center x and y of the sunspot
        sunspotX = np.mean(bcol)
        sunspotY = np.mean(brow)

        sunspot_blob.append((sunspotX, sunspotY, numPixels))

    # Gets the largest sunspot 
    def get_largest(suns):
        return -suns[2]
    
    # Sorts making the largest sunspot by size first
    # Helps to track 4366 in the Dataset
    sunspot_blob.sort(key=get_largest)

    return sunspot_blob

Gets the data needed from 4366, the largest sunspot we are tracking

In [ ]:
image = {}
param = {}
sunspots = {}

for i in range(1, Days + 1):
    # Path to each image
    path = f'{IMAGE}/Feb{i},jpg'

    # Opens the Jpeg with PIl
    # Converts to an np.array
    # 1024x1024 with RGB channels
    img = np.array(Image.open(path))

    image[d] = img

    x,y,R = getCenter(img)

    param[d] = (x,y,R)

    sunspots[d] = getSunspots(img,x,y,R)

tracked = {}

for day in range(1,Days+1):
    largestSunspot = sunspots[day][0]
    # Gets the X and Y coord of sunspot 4366
    tracked[day] = (largestSunspot[0], largestSunspot[1])

# Print a table of the tracked positions for inspection
print("\n4366 tracked positions:")
print(f"{'Day':<6} {'X':>9} {'Y':>9}")
print("-" * 28)
for day in range(1,Days+1):
    xp,yp = tracked[day]
    print(f"Feb {d:<3} {xp:>9.1f} {yp:>9.1f}")


days = np.array(sorted(tracked.keys()),dtype=float)

# Date Strings for axis tick label on plots
dayLabel = [f'Feb {d:.0f}' for day in days]


Gets coordinates of the sunspot. Since the Sun is not 2d, we need to account for geometry.

In [ ]:
'''
** Assistance from AI **
Formulas needed: 
dx = x - xCenter
dy = yCenter - y
xi = dx/ R
yi = dy/ R
dc = sqrt(xi^2 + yi^2)

Latitude: lat = arcsin(yi)
Longtitude:: lon = arcsin(xi/cos(lat))
'''

longitude = []
latitude = []
distCen = []

for day in days.astype(int):
    # Suns center and radius
    xs,ys,R = getCenter(day)
    # 4366 coordinates
    x,y = tracked[d]

    # Horizontal Pixel offset from Center of the sun
    dx = x - xs

    # Vertical pixel offset from Center of sun
    dy = ys - y

    xi = dx / R
    yi = dy / R

    # How far sunspot is from center
    dc = np.sqrt(xi**2 + yi**2)

    # Converts the yi to heliographic latitude
    # keeps in range of [-1,1]
    # Gets the latitude 
    lat = np.degrees(np.arcsin(np.clip(yi, -1, 1)))

    # Gets longitude by converting xi
    lon = np.degrees(np.arcsin(np.clip(xi/no.cos(np.radians(lat)),-1,1)))

    longitude.append(lon)
    latitude.append(lat)
    distCen.append(dc)

    # Convert to np.arr
    longitude = np.array(longitude)
    latitude = np.array(latitude)
    distCen = np.array(distCen)